# Ders 03 - Agentik Tasarım Kalıpları

Bu derste, etkili AI ajanları oluşturmak için üç temel tasarım kalıbını keşfedeceğiz:

1. **Net Ajan Talimatları** — Ajan davranışını yönlendiren kesin, rol tanımlayıcı istemler hazırlama
2. **Pydantic Modelleri ile Yapılandırılmış Çıktı** — Ajanların tahmin edilebilir, doğrulanmış veriler döndürmesini sağlama
3. **Tek Sorumluluk Ajanları** — Her biri bir işi iyi yapan odaklanmış ajanlar tasarlama

Her kalıbı, destinasyon öneren bir **seyahat destinasyonu önericisi** senaryosuna uygulayacağız ve giderek destinasyon önerebilen, müsaitliği kontrol eden ve lojistiği yöneten bir sistem kuracağız.


## Kurulum


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Desen 1: Net Ajan Talimatları

En etkili desen aynı zamanda en basit olanıdır: ajanın için net, detaylı talimatlar yazmak.

İyi talimatlar şunları tanımlar:
- **Kim** ajanın (kişilik ve ton)
- **Ne** yapması gerektiği (adım adım sorumluluklar)
- **Nasıl** davranması gerektiği (kısıtlamalar ve stil)

Aşağıda, ürettiği her cevabı şekillendiren açık talimatlarla bir seyahat konsiyerj ajanı oluşturuyoruz.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic Modelleri ile Yapılandırılmış Çıktı

Serbest biçimli metin sohbet için faydalıdır, ancak alt sistemlerin yapılandırılmış verilere ihtiyacı vardır.  
**Pydantic modelleri** ile bir **araç fonksiyonunu** eşleştirerek şunları yapabiliriz:

- Ajanın çıktısı için kesin bir şema tanımlamak
- Yanıtları otomatik olarak doğrulamak
- Ajan sonuçlarını uygulama mantığına güvenilir şekilde entegre etmek

Ayrıca, ajanın önerilerini gerçek verilere dayandırmasını sağlamak için varış noktası detaylarını döndüren bir araç tanıtıyoruz.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Desen 3: Tek Sorumluluklu Ajanlar

Karmaşık görevler, her biri tek bir sorumluluğa sahip birden fazla odaklanmış ajana iş bölümü yaparak fayda sağlar:

- Yerler ve müsaitlik hakkında bilgi sahibi bir **Varış Yeri Uzmanı**
- Uçuşlar, oteller ve güzergahları yöneten bir **Lojistik Planlayıcı**

Bu, yazılım mühendisliği prensibi olan *kaygıların ayrılması* ile paralellik arz eder — her ajan bağımsız olarak test edilmesi, bakımı ve geliştirilmesi daha kolaydır.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Özet

Bu derste, seyahat öneri senaryosuna üç ajan tasarım kalıbı uyguladık:

| Kalıp | Ana Fikir | Faydası |
|---|---|---|
| **Net Talimatlar** | Kişilik, sorumluluklar ve kısıtlamaları önceden tanımla | Tutarlı, marka uyumlu ajan davranışı |
| **Yapılandırılmış Çıktı** | Yanıt formatı olarak Pydantic modellerini kullan | Doğrulanmış, makine tarafından okunabilir sonuçlar |
| **Tek Sorumluluk** | Her ajana tek bir odaklanmış görev ver | Test etmesi, bakımı ve birleştirmesi daha kolay |

Bu kalıplar doğal olarak birleşir — net talimatları yapılandırılmış çıktı ile tek sorumluluk ajanı içinde birleştirerek sağlam, üretim hazır sistemler inşa edebilirsiniz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:  
Bu belge, [Co-op Translator](https://github.com/Azure/co-op-translator) AI çeviri hizmeti kullanılarak çevrilmiştir. Doğruluk için çaba gösterilse de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi ana dilindeki haliyle yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalar veya yanlış yorumlamalar nedeniyle sorumluluk kabul edilmemektedir.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
